In [0]:
from pyspark.sql import functions as F

In [0]:

df = spark.table("skztour.01_bronze.raw_wikipedia_maniac_5star")
display(df)

Date_2023,City,Country,Venue,Attendance
"April 29, 2022",Seoul,South Korea,Jamsil Arena Beyond Live [ A ],—
"April 30, 2022",Seoul,South Korea,Jamsil Arena Beyond Live [ A ],—
"May 1, 2022",Seoul,South Korea,Jamsil Arena Beyond Live [ A ],—
"June 11, 2022",Kobe,Japan,World Memorial Hall,—
"June 12, 2022",Kobe,Japan,World Memorial Hall,—
"June 18, 2022",Tokyo,Japan,Yoyogi National Gymnasium,"24,000 [ 48 ]"
"June 19, 2022",Tokyo,Japan,Yoyogi National Gymnasium,"24,000 [ 48 ]"
"June 28, 2022",Newark,United States,Prudential Center,—
"June 29, 2022",Newark,United States,Prudential Center,—
"July 1, 2022",Chicago,United States,United Center,—


In [0]:

# 1. Remove referências [ G ], [ A ] das datas
# 2. Converter data para date type
# 3. Remover referências dos outros campos
# 4. Limpar attendance (remover [ 11 ], etc)

print("📊 Verificando formatos de data originais:")
df.select("Date_2023").show(10, truncate=False)

df_clean = df \
    .withColumn("event_date", F.trim(F.regexp_replace("Date_2023", r"\[.*?\]", ""))) \
    .withColumn("source", F.lit("wikipedia_maniac_5star")) \
    .withColumn("venue_name", F.trim(F.regexp_replace("Venue", r"\[.*?\]", ""))) \
    .withColumn("attendance", F.trim(F.regexp_replace("Attendance", r"\[.*?\]", ""))) \
    .withColumn("venue_city", F.trim(F.regexp_replace("City", r"\[.*?\]", ""))) \
    .withColumn("attendance", F.when(F.col("attendance") == "—", None).otherwise(F.col("attendance")))

print("\n🔍 Datas após limpeza de referências:")
df_clean.select("event_date").show(10, truncate=False)

# ============================================
# LÓGICA CONDICIONAL: Adicionar ano SOMENTE se não existir
# ============================================
# Técnica: Verificar se a string contém vírgula
#   - "April 29, 2022" → JÁ tem ano (tem vírgula)
#   - "August 24" → NÃO tem ano (sem vírgula)

df_with_year = df_clean.withColumn(
    "date_with_year",
    F.when(
        F.col("event_date").contains(","),  # Se JÁ tem vírgula (ano presente)
        F.col("event_date")                   # Mantém como está
    ).otherwise(
        F.concat(F.col("event_date"), F.lit(", 2023"))  # Adiciona ", 2023"
    )
)

print("Datas com ano adicionado condicionalmente:")
df_with_year.select("event_date").show(10, truncate=False)

# Converter para DATE type
df_final = df_with_year \
    .withColumn("event_date", F.to_date(F.col("date_with_year"), "MMMM d, yyyy")) \
    .withColumn("event_year", F.year("event_date")) \
    .select(
        F.monotonically_increasing_id().cast("string").alias("show_id"),
        "source",
        F.col("event_date").alias("event_date"),
        "event_year",
        F.lit("Stray Kids").alias("artist_name"),
        F.col("venue_name").alias("venue_name"),
        F.col("venue_city").alias("venue_city"),
        F.col("Country").alias("country"),
        F.regexp_replace(F.col("attendance"), ",", "").cast("int").alias("attendance"),
        F.lit(None).cast("string").alias("sets_json"),
        F.lit(None).cast("string").alias("info"),
        F.lit("https://en.wikipedia.org/wiki/Maniac_World_Tour").alias("source_url")
    )

# Armazenar count para reusar (evita recomputação)
total_shows = df_final.count()
print(f"Total shows: {total_shows}")
df_final.show(5, truncate=False)

📊 Verificando formatos de data originais:
+--------------+
|Date_2023     |
+--------------+
|April 29, 2022|
|April 30, 2022|
|May 1, 2022   |
|June 11, 2022 |
|June 12, 2022 |
|June 18, 2022 |
|June 19, 2022 |
|June 28, 2022 |
|June 29, 2022 |
|July 1, 2022  |
+--------------+
only showing top 10 rows

🔍 Datas após limpeza de referências:
+--------------+
|event_date    |
+--------------+
|April 29, 2022|
|April 30, 2022|
|May 1, 2022   |
|June 11, 2022 |
|June 12, 2022 |
|June 18, 2022 |
|June 19, 2022 |
|June 28, 2022 |
|June 29, 2022 |
|July 1, 2022  |
+--------------+
only showing top 10 rows
Datas com ano adicionado condicionalmente:
+--------------+
|event_date    |
+--------------+
|April 29, 2022|
|April 30, 2022|
|May 1, 2022   |
|June 11, 2022 |
|June 12, 2022 |
|June 18, 2022 |
|June 19, 2022 |
|June 28, 2022 |
|June 29, 2022 |
|July 1, 2022  |
+--------------+
only showing top 10 rows
Total shows: 52
+-------+----------------------+----------+----------+-----------+------

In [0]:
print("Cidades únicas no dataset:")
df_final.select("venue_city").distinct().orderBy("venue_city").show(50, truncate=False)

Cidades únicas no dataset:
+-----------+
|venue_city |
+-----------+
|Anaheim    |
|Atlanta    |
|Chicago    |
|Fort Worth |
|Fukuoka    |
|Inglewood  |
|Jakarta    |
|Kobe       |
|Los Angeles|
|Manila     |
|Melbourne  |
|Nagoya     |
|Newark     |
|Oakland    |
|Osaka      |
|Pak Kret   |
|Saitama    |
|Seattle    |
|Seoul      |
|Singapore  |
|Sydney     |
|Tokyo      |
+-----------+



In [0]:
# ============================================
# VALIDAÇÃO DE QUALIDADE DOS DADOS
# ============================================

print("🔍 Verificando qualidade dos dados:")
print("="*50)

# 1. Verificar datas nulas após parsing
null_dates = df_final.filter(F.col("event_date").isNull()).count()
print(f"\n• Datas nulas: {null_dates}")
if null_dates > 0:
    print("  ⚠️  ATENÇÃO: Algumas datas não foram parseadas corretamente!")
    df_final.filter(F.col("event_date").isNull()).select("Date_2023", "event_date").show(truncate=False)

# 2. Verificar duplicatas por show_id
duplicates = df_final.groupBy("show_id").count().filter(F.col("count") > 1).count()
print(f"\n• Show IDs duplicados: {duplicates}")

# 3. Verificar valores vazios em campos críticos
critical_fields = ["event_date", "venue_name", "venue_city", "country"]
for field in critical_fields:
    null_count = df_final.filter(F.col(field).isNull()).count()
    print(f"\n• {field} nulos: {null_count}")

# 4. Range de datas
date_range = df_final.agg(
    F.min("event_date").alias("primeira_data"),
    F.max("event_date").alias("ultima_data")
).collect()[0]

print(f"\n• Range de datas: {date_range['primeira_data']} a {date_range['ultima_data']}")

# 5. Distribuição por ano
print("\n• Shows por ano:")
df_final.groupBy("event_year").count().orderBy("event_year").show()

print("="*50)
print("✅ Validação concluída!\n")

🔍 Verificando qualidade dos dados:

• Datas nulas: 0

• Show IDs duplicados: 0

• event_date nulos: 0

• venue_name nulos: 0

• venue_city nulos: 0

• country nulos: 0

• Range de datas: 2022-04-29 a 2023-10-29

• Shows por ano:
+----------+-----+
|event_year|count|
+----------+-----+
|      2022|   23|
|      2023|   29|
+----------+-----+

✅ Validação concluída!



In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("skztour.02_silver.clean_maniac_5star")

print("Tabela clean_maniac_5star salva com sucesso!")
print(f"Total de shows salvos: {total_shows}")

Tabela clean_maniac_5star salva com sucesso!
Total de shows salvos: 52


In [0]:
# Verificar resultado final com schema padronizado
print("Schema final:")
df_final.printSchema()

Schema final:
root
 |-- show_id: string (nullable = false)
 |-- source: string (nullable = false)
 |-- event_date: date (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- artist_name: string (nullable = false)
 |-- venue_name: string (nullable = true)
 |-- venue_city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- attendance: integer (nullable = true)
 |-- sets_json: string (nullable = true)
 |-- info: string (nullable = true)
 |-- source_url: string (nullable = false)



In [0]:
display(df_final)

show_id,source,event_date,event_year,artist_name,venue_name,venue_city,country,attendance,sets_json,info,source_url
0,wikipedia_maniac_5star,2022-04-29,2022,Stray Kids,Jamsil Arena Beyond Live,Seoul,South Korea,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
1,wikipedia_maniac_5star,2022-04-30,2022,Stray Kids,Jamsil Arena Beyond Live,Seoul,South Korea,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
2,wikipedia_maniac_5star,2022-05-01,2022,Stray Kids,Jamsil Arena Beyond Live,Seoul,South Korea,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
3,wikipedia_maniac_5star,2022-06-11,2022,Stray Kids,World Memorial Hall,Kobe,Japan,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
4,wikipedia_maniac_5star,2022-06-12,2022,Stray Kids,World Memorial Hall,Kobe,Japan,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
5,wikipedia_maniac_5star,2022-06-18,2022,Stray Kids,Yoyogi National Gymnasium,Tokyo,Japan,24000,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
6,wikipedia_maniac_5star,2022-06-19,2022,Stray Kids,Yoyogi National Gymnasium,Tokyo,Japan,24000,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
7,wikipedia_maniac_5star,2022-06-28,2022,Stray Kids,Prudential Center,Newark,United States,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
8,wikipedia_maniac_5star,2022-06-29,2022,Stray Kids,Prudential Center,Newark,United States,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
9,wikipedia_maniac_5star,2022-07-01,2022,Stray Kids,United Center,Chicago,United States,null,null,null,https://en.wikipedia.org/wiki/Maniac_World_Tour
